# INF01017 - Aprendizado de Máquina
## Atividade Prática : Interpretabilidade de Modelos — Regressão

---

## Sobre esta atividade

Este notebook apresenta um **exemplo guiado completo** de treinamento e interpretação de modelos de aprendizado de máquina para um problema de **regressão**, utilizando o dataset `fetch_california_housing()`.

A ideia é que você:

1. **Compreenda a estrutura geral do pipeline**, incluindo:
   - treinamento dos modelos (Decision Tree, Regressão Linear e LightGBM)
   - avaliação de desempenho (RMSE e R²)
   - aplicação de técnicas de interpretabilidade (Permutation Importance, PDP e SHAP)

2. **Analise cuidadosamente os resultados obtidos**, especialmente:
   - quais atributos têm mais impacto sobre o preço mediano das casas
   - como diferentes técnicas explicam cada modelo
   - diferenças entre modelos interpretáveis e de caixa-preta

---

## Contexto do dataset

O **California Housing dataset** contém informações sobre blocos residenciais do estado da Califórnia (censo de 1990).  
O objetivo é prever o **valor mediano das casas** (em centenas de milhares de dólares).

| Feature | Descrição |
|---|---|
| `MedInc` | Renda mediana do bloco (em dezenas de milhares de dólares) |
| `HouseAge` | Idade mediana das casas do bloco |
| `AveRooms` | Número médio de cômodos por residência |
| `AveBedrms` | Número médio de quartos por residência |
| `Population` | População do bloco |
| `AveOccup` | Número médio de moradores por residência |
| `Latitude` | Latitude do bloco |
| `Longitude` | Longitude do bloco |

**Variável alvo:** `MedHouseVal` — valor mediano das casas (em centenas de milhares de dólares, limitado a 5.0).

---


## 1. Importação de bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, r2_score

from lightgbm import LGBMRegressor
import shap

import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento do dataset

Carregamos o California Housing dataset e o convertemos em um DataFrame para facilitar a manipulação.


In [ ]:
import pandas as pd
from sklearn.utils import Bunch

# Carrega os dados a partir dos arquivos csv. Isto foi alterado pois o download
# automático pelo Python estava falhando.
# Faça o Download em: https://drive.google.com/drive/folders/1S95I3UxS_4ZO-BKN9R72g7bHKLbMo2Uq?usp=sharing
# Ajuste o caminho nos comandos read_csv() abaixo
train_df = pd.read_csv('/content/sample_data/california_housing_train.csv')
test_df = pd.read_csv('/content/sample_data/california_housing_test.csv')

# Combina ambos os arquivos para recriar o dataset completo
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Define as features (X) e o alvo (y)
# No CSV local, a coluna alvo chama-se 'median_house_value' e os valores estão na escala original
# Vamos dividir por 100.000 para ficar na mesma escala do fetch_california_housing (centenas de milhares de dólares)
X = full_df.drop(columns=['median_house_value'])
y = full_df['median_house_value'] / 100000.0
y.name = "MedHouseVal"

# Ajusta os nomes das colunas locais para bater exatamente com os nomes esperados pelo scikit-learn
X = pd.DataFrame({
    'MedInc': X['median_income'],
    'HouseAge': X['housing_median_age'],
    'AveRooms': X['total_rooms'] / X['households'],
    'AveBedrms': X['total_bedrooms'] / X['households'],
    'Population': X['population'],
    'AveOccup': X['households'],
    'Latitude': X['latitude'],
    'Longitude': X['longitude']
})

# Para compatibilidade com outras partes do código que usam o objeto 'data'
data = Bunch(
    data=X.values,
    target=y.values,
    feature_names=list(X.columns),
    target_names=['MedHouseVal']
)

print(f"Dimensões: {X.shape[0]} instâncias, {X.shape[1]} features")
print(f"\nVariável alvo — MedHouseVal:")
print(f"  min={y.min():.2f}, max={y.max():.2f}, média={y.mean():.2f}, mediana={y.median():.2f}")
X.head()

In [ ]:
# Estatísticas descritivas
X.describe().round(2)

## 3. Divisão treino / teste

Separamos **20 %** dos dados como conjunto de teste final — esses dados **não são tocados** durante a seleção de hiperparâmetros nem o treino.  
Os **80 % restantes** são usados inteiramente dentro do Nested Cross-Validation.

> **Atenção:** as técnicas de interpretabilidade (Permutation Importance, PDP e SHAP) são calculadas no **conjunto de teste** (Permutation Importance) e no **conjunto de treino completo** (PDP), conforme recomendado por Molnar (2024).

In [ ]:
# Separa 20 % para teste final — esses dados não são usados em nenhuma etapa anterior
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Treino+CV: {X_train_full.shape[0]} instâncias ({X_train_full.shape[0]/len(X)*100:.0f} %)")
print(f"Teste:     {X_test.shape[0]} instâncias ({X_test.shape[0]/len(X)*100:.0f} %)")

## 4. Treinamento dos modelos

Comparamos três modelos com diferentes graus de interpretabilidade:

| Modelo | Interpretabilidade intrínseca |
|---|---|
| **Decision Tree Regressor** | Alta — estrutura de árvore diretamente inspecionável |
| **Regressão Linear** | Alta — coeficientes são diretamente interpretáveis |
| **LightGBM (GBM)** | Baixa — ensemble de centenas de árvores (caixa-preta) |

### 4.1 Seleção da profundidade da Decision Tree via Nested Cross-Validation

O **Nested CV** evita o viés de seleção do hold-out simples: o loop externo estima o desempenho real de cada configuração, enquanto o loop interno escolhe o melhor hiperparâmetro.

```
Loop externo  (5 folds) → estima RMSE médio de cada profundidade
Loop interno  (3 folds) → seleciona a profundidade por fold externo
```

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

# Hiperparâmetros a testar
param_grid = {"max_depth": [2, 3, 4, 5, 6, 7]}

# Loop externo: 5 folds para estimar o RMSE de cada profundidade
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Loop interno: 3 folds para escolher a melhor profundidade dentro de cada fold externo
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

# GridSearchCV encapsula o loop interno
grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_root_mean_squared_error",
    refit=True          # re-treina no fold completo com o melhor param
)

# cross_val_score executa o loop externo
nested_scores = cross_val_score(
    grid_search,
    X_train_full, y_train_full,
    cv=outer_cv,
    scoring="neg_root_mean_squared_error"
)

nested_rmse = -nested_scores
print("RMSE por fold externo:", nested_rmse.round(4))
print(f"RMSE médio (Nested CV): {nested_rmse.mean():.4f} ± {nested_rmse.std():.4f}")

# Re-ajusta o GridSearchCV em todo X_train_full para obter o best_depth final
grid_search.fit(X_train_full, y_train_full)
best_depth = grid_search.best_params_["max_depth"]
print(f"\nMelhor max_depth selecionado (loop interno, dados completos): {best_depth}")

### 4.2 Treinamento final (80 % completos)

In [ ]:
# Treino final: usa todo X_train_full (os 80 %) com o best_depth do Nested CV

# --- Decision Tree ---
tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
tree.fit(X_train_full, y_train_full)

# --- Regressão Linear ---
lr = LinearRegression()
lr.fit(X_train_full, y_train_full)

# --- LightGBM ---
lgbm = LGBMRegressor(n_estimators=300, max_depth=best_depth, random_state=42,
                     verbose=-1)
lgbm.fit(X_train_full, y_train_full)

print("Modelos treinados com sucesso.")
print(f"  Decision Tree  → max_depth={best_depth}")
print(f"  Regressão Linear → sem hiperparâmetro de profundidade")
print(f"  LightGBM       → max_depth={best_depth}")

## 5. Avaliação de desempenho no conjunto de teste

Para regressão, usamos duas métricas complementares:

- **RMSE** (*Root Mean Squared Error*): erro médio em unidades da variável alvo (centenas de milhares de dólares). Penaliza erros grandes.
- **R²** (*coeficiente de determinação*): fração da variância explicada pelo modelo. Varia de −∞ a 1 (1 = previsão perfeita).


In [ ]:
models = {
    "Decision Tree":    tree,
    "Regressão Linear": lr,
    "LightGBM":        lgbm,
}

print(f"{'Modelo':<22} {'RMSE':>8} {'R²':>8}")
print("-" * 40)
for name, model in models.items():
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    print(f"{name:<22} {rmse:>8.4f} {r2:>8.4f}")

In [ ]:
# Visualização: predito vs real para os três modelos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    ax.scatter(y_test, y_pred, alpha=0.3, s=8, color="steelblue")
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, "r--", lw=1.5, label="Ideal")
    ax.set_xlabel("MedHouseVal real")
    ax.set_ylabel("MedHouseVal predito")
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle("Predito vs Real — Conjunto de Teste", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 6. Permutation Importance

A **Permutation Importance** mede o quanto o desempenho do modelo cai quando os valores de uma feature são embaralhados aleatoriamente — destruindo qualquer relação entre ela e o alvo.

> **Por que no conjunto de teste?**  
> Queremos medir a importância das features em dados nunca vistos, refletindo o comportamento real do modelo em produção.

A técnica é **agnóstica ao modelo**: funciona da mesma forma para a Decision Tree, a Regressão Linear e o LightGBM.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

perm_results = {}

for ax, (name, model) in zip(axes, models.items()):
    result = permutation_importance(
        model, X_test, y_test,
        n_repeats=10, random_state=42,
        scoring="neg_root_mean_squared_error"
    )
    perm_results[name] = result

    # Ordena por importância média
    sorted_idx = np.argsort(result.importances_mean)

    ax.barh(X.columns[sorted_idx], result.importances_mean[sorted_idx],
            xerr=result.importances_std[sorted_idx],
            color="steelblue", alpha=0.8, capsize=4)
    ax.set_title(f"Permutation Importance\n({name})")
    ax.set_xlabel("Aumento médio no RMSE")
    ax.axvline(0, color="gray", linestyle="--", lw=0.8)

plt.suptitle("Permutation Importance — Conjunto de Teste", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Interpretação

- Cada barra representa o aumento médio no RMSE quando aquela feature é embaralhada.
- **Barras longas** → feature muito importante: removê-la piora bastante o modelo.
- **Barras curtas (ou negativas)** → feature pouco importante: o modelo não depende muito dela.
- As barras de erro mostram a variabilidade entre as repetições do embaralhamento.

> **Questão para reflexão:** os três modelos concordam sobre quais features são mais importantes?


## 7. Partial Dependence Plots (PDP)

O **PDP** mostra a relação marginal entre uma feature e a variável alvo, marginalizando sobre todas as outras features.

> **Por que usar os dados de treino?**  
> Conforme Molnar (2024), a estimativa do PDP é formalmente definida como uma média sobre a distribuição dos **dados de treino** (`X_train_full`). Usar o conjunto de teste aqui seria matematicamente inconsistente com a definição — além de desperdiçar o sinal de distribuição disponível nos 80 % de dados.

> **Interpretação:** o eixo Y representa a **mudança esperada na predição** ao variar o valor da feature, mantendo as demais em seus valores observados no treino.

In [ ]:
# Seleciona os top-3 atributos de cada modelo (por permutation importance)
top3 = {}
for name, result in perm_results.items():
    top3[name] = list(np.argsort(result.importances_mean)[-3:])
    print(f"Top 3 — {name}: {list(X.columns[top3[name]])}")

In [ ]:
# PDP para cada modelo
# Conforme Molnar (2024, cap. 19), o PDP marginaliza sobre a distribuição
# dos dados de TREINO — usamos X_train_full para representar essa distribuição.
for name, model in models.items():
    fig, ax = plt.subplots(figsize=(11, 4))

    PartialDependenceDisplay.from_estimator(
        model,
        X_train_full,          # ← treino, conforme Molnar
        features=top3[name],
        feature_names=X.columns.tolist(),
        ax=ax,
        random_state=42
    )
    plt.suptitle(f"PDP — {name}", fontsize=13)
    plt.tight_layout()
    plt.show()

### Interpretação

- **Curva crescente:** valores maiores dessa feature tendem a aumentar a predição do preço.
- **Curva decrescente:** valores maiores tendem a diminuir o preço previsto.
- **Curva em degrau (Decision Tree):** reflexo da estrutura de nós — a árvore só produz predições constantes por folha.
- **Curva suave (LightGBM / Linear):** relação mais contínua e gradual.

> **Atenção:** o PDP assume que as features são **independentes entre si**. Em datasets com correlações fortes (ex: `AveRooms` e `AveBedrms`), o gráfico pode ser enganoso.


## 8. SHAP — SHapley Additive exPlanations

SHAP é baseado na teoria dos jogos cooperativos: atribui a cada feature uma contribuição justa para a diferença entre a predição de uma instância e a predição média do modelo.

**Vantagens sobre outras técnicas:**
- Consistente e fundamentado matematicamente
- Funciona tanto em nível global (todo o dataset) quanto local (uma instância)
- Revela a **direção** do efeito (positivo ou negativo na predição)

### 8.1 SHAP Global — Summary Plot (Beeswarm)


In [ ]:
# --- Decision Tree ---
explainer_tree = shap.TreeExplainer(tree)
shap_values_tree = explainer_tree.shap_values(X_test)

print("=== Decision Tree ===")
shap.summary_plot(shap_values_tree, X_test, feature_names=X.columns, show=True)

In [ ]:
# --- LightGBM ---
explainer_lgbm = shap.TreeExplainer(lgbm)
shap_values_lgbm = explainer_lgbm.shap_values(X_test)

print("=== LightGBM ===")
shap.summary_plot(shap_values_lgbm, X_test, feature_names=X.columns, show=True)

In [ ]:
# --- Regressão Linear (usa LinearExplainer) ---
explainer_lr = shap.LinearExplainer(lr, X_train_full)
shap_values_lr = explainer_lr.shap_values(X_test)

print("=== Regressão Linear ===")
shap.summary_plot(shap_values_lr, X_test, feature_names=X.columns, show=True)

### Interpretação do Summary Plot (Beeswarm)

- Cada ponto representa **uma instância** do conjunto de teste.
- **Posição horizontal (eixo X):** valor SHAP — quanto aquela feature contribuiu para elevar (+) ou reduzir (−) a predição, em relação à predição média.
- **Cor:** valor original da feature (vermelho = alto, azul = baixo).

> **Exemplo de leitura:** se `MedInc` tem muitos pontos vermelhos à direita, significa que rendas altas tendem a **aumentar** a predição do preço.


### 8.2 SHAP Bar Plot — Importância Global

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, sv) in zip(axes, [
    ("Decision Tree",    shap_values_tree),
    ("Regressão Linear", shap_values_lr),
    ("LightGBM",         shap_values_lgbm),
]):
    mean_abs_shap = np.abs(sv).mean(axis=0)
    sorted_idx = np.argsort(mean_abs_shap)
    ax.barh(X.columns[sorted_idx], mean_abs_shap[sorted_idx],
            color="coral", alpha=0.85)
    ax.set_title(f"SHAP — {name}")
    ax.set_xlabel("|SHAP| médio")

plt.suptitle("Importância Global via SHAP (|valores SHAP| médios)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 8.3 SHAP Local — Análise de instâncias individuais

In [ ]:
# Escolha uma instância para analisar
idx = 42  # TODO: teste outros índices

print(f"Instância #{idx} — valores das features:")
print(X_test.iloc[idx].to_string())
print(f"\nValor real:            {y_test.iloc[idx]:.3f}")
print(f"Predição (Tree):       {tree.predict(X_test.iloc[[idx]])[0]:.3f}")
print(f"Predição (Linear):     {lr.predict(X_test.iloc[[idx]])[0]:.3f}")
print(f"Predição (LightGBM):   {lgbm.predict(X_test.iloc[[idx]])[0]:.3f}")

In [ ]:
# Waterfall plot — LightGBM (instância idx)
print("=== LightGBM — Waterfall Plot ===")
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_lgbm[idx],
        base_values=explainer_lgbm.expected_value,
        data=X_test.iloc[idx],
        feature_names=X.columns.tolist()
    )
)

In [ ]:
# Waterfall plot — Decision Tree (instância idx)
print("=== Decision Tree — Waterfall Plot ===")
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_tree[idx],
        base_values=explainer_tree.expected_value,
        data=X_test.iloc[idx],
        feature_names=X.columns.tolist()
    )
)

### Interpretação do Waterfall Plot

- Começa na **predição média** do modelo (`E[f(X)]`).
- Cada barra representa a **contribuição de uma feature** para empurrar a predição acima (vermelho/laranja) ou abaixo (azul) da média.
- Termina na **predição final** para aquela instância.

> **Para reflexão:** os dois modelos chegam na mesma predição final? As features mais influentes são as mesmas?


## 9. Modelos intrinsecamente interpretáveis

### 9.1 Visualização da Decision Tree


In [ ]:
plt.figure(figsize=(20, 6))
plot_tree(
    tree,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=4          # limita visualização para as primeiras 3 camadas
)
plt.title(f"Decision Tree Regressor (max_depth={best_depth}) — primeiros 4 níveis", fontsize=13)
plt.tight_layout()
plt.show()

**Como ler a árvore:**
- Cada nó interno mostra: a **condição de divisão**, o MSE médio das instâncias naquele nó, e o número de amostras.
- Cada nó folha mostra o **valor predito** (média das instâncias que chegam ali).
- A intensidade da cor representa o valor predito (mais escuro = maior preço).


### 9.2 Coeficientes da Regressão Linear

In [ ]:
coef_df = pd.DataFrame({
    "Feature":     X.columns,
    "Coeficiente": lr.coef_
}).sort_values("Coeficiente", key=abs, ascending=False)

print(f"Intercepto: {lr.intercept_:.4f}\n")
print(coef_df.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["steelblue" if c > 0 else "tomato" for c in coef_df["Coeficiente"]]
ax.barh(coef_df["Feature"], coef_df["Coeficiente"], color=colors, alpha=0.85)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Coeficiente")
ax.set_title("Coeficientes — Regressão Linear")
plt.tight_layout()
plt.show()

**Atenção:** os coeficientes da Regressão Linear são interpretáveis diretamente, mas apenas quando as features estão na **mesma escala**. Comparar coeficientes de features em escalas diferentes pode ser enganoso.

> Para uma comparação justa, teríamos que padronizar as features antes do treino. Mesmo assim, os valores SHAP (Seção 8) já oferecem uma medida de importância comparável e robusta.


## 10. Comparação entre técnicas de interpretabilidade

A tabela abaixo resume as características de cada técnica aplicada:

| Técnica | Nível | Depende do modelo? | Direção do efeito? | Pressuposto |
|---|---|---|---|---|
| **Permutation Importance** | Global | Não (agnóstico) | Não | Features independentes |
| **PDP** | Global | Não (agnóstico) | Sim | Features independentes |
| **SHAP** | Global + Local | Parcialmente | Sim | Consistência matemática |
| **Coeficientes (Linear)** | Global | Sim | Sim | Linearidade + mesma escala |
| **Estrutura da árvore** | Global + Local | Sim | Sim | Nenhum |

### 10.1 Ranking consolidado de importância


In [ ]:
# Tabela comparativa: Permutation Importance + SHAP para os 3 modelos
methods = [
    ("Perm. Imp. — Tree",   perm_results["Decision Tree"].importances_mean),
    ("Perm. Imp. — Linear", perm_results["Regressão Linear"].importances_mean),
    ("Perm. Imp. — LightGBM", perm_results["LightGBM"].importances_mean),
    ("SHAP — Tree",    np.abs(shap_values_tree).mean(axis=0)),
    ("SHAP — Linear",  np.abs(shap_values_lr).mean(axis=0)),
    ("SHAP — LightGBM", np.abs(shap_values_lgbm).mean(axis=0)),
]

# Normaliza cada método para [0,1] para comparação relativa
ranking_df = pd.DataFrame(index=X.columns)
for label, values in methods:
    v = np.array(values)
    v_norm = (v - v.min()) / (v.max() - v.min() + 1e-12)
    ranking_df[label] = v_norm

print("Importância relativa (normalizada 0→1):")
print(ranking_df.round(3).to_string())

In [ ]:
# Heatmap de importâncias relativas
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(ranking_df.T.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(X.columns)))
ax.set_xticklabels(X.columns, rotation=30, ha="right")
ax.set_yticks(range(len(ranking_df.columns)))
ax.set_yticklabels(ranking_df.columns)
plt.colorbar(im, ax=ax, label="Importância relativa (normalizada)")
ax.set_title("Heatmap de importâncias — comparação entre técnicas e modelos")
plt.tight_layout()
plt.show()